# Bulk RNA-seq Explorer

파이프라인 HTML 리포트의 인터랙티브 보완본. 각 섹션은 `report/sections/_*.qmd`의 코드를 그대로 미러링하므로 딜리버러블과 동일한 플롯이 재생성됨 — cutoff, 색상, 유전자 라벨, 플롯 호출부를 수정하고 재실행하면 Snakemake 재실행 없이 반복 가능.

리포트의 contrast별 루프는 notebook에선 단일 contrast로 unroll되어 있음 — Setup 셀의 `CONTRAST`를 변경해 전환.

## Setup

In [ ]:
suppressPackageStartupMessages({
  library(yaml); library(readr); library(dplyr); library(tidyr)
  library(tibble); library(ggplot2); library(scales)
})

`%||%` <- function(a, b) if (is.null(a)) b else a

PROJECT_ROOT <- Sys.getenv("PROJECT_ROOT", normalizePath("..", mustWork = TRUE))
CONFIG_PATH  <- file.path(PROJECT_ROOT, "config", "config.yaml")

rp <- function(...) file.path(PROJECT_ROOT, ...)

cfg       <- yaml::read_yaml(CONFIG_PATH)
samples   <- read_tsv(rp(cfg$input$samples_tsv   %||% "config/samples.tsv"),   show_col_types = FALSE)
contrasts <- read_tsv(rp(cfg$input$contrasts_tsv %||% "config/contrasts.tsv"), show_col_types = FALSE)

no_results <- function(msg = "No results available for this section.") {
  message(msg); invisible(NULL)
}

safe_read <- function(path, sep = ",") {
  if (is.null(path) || !file.exists(path)) return(NULL)
  df <- tryCatch(
    if (sep == "\t") read_tsv(path, show_col_types = FALSE)
    else             read_csv(path, show_col_types = FALSE),
    error = function(e) NULL)
  if (is.null(df) || nrow(df) == 0) return(df)
  df
}

contrast_row <- function(cid) {
  out <- contrasts[contrasts$contrast_id == cid, , drop = FALSE]
  if (nrow(out) == 0) return(NULL)
  as.list(out[1, ])
}

# 탐색할 contrast 하나 선택 — 변경 후 하위 셀 재실행.
CONTRAST <- contrasts$contrast_id[1]
cat("Project :", basename(PROJECT_ROOT),
    "\nContrast:", CONTRAST,
    "\nAvailable:", paste(contrasts$contrast_id, collapse = ", "), "\n")

## 1. QC

MultiQC에서 집계한 샘플별 QC 지표 (`results/qc/qc_summary.tsv`).

In [ ]:
qc <- safe_read(rp("results", "qc", "qc_summary.tsv"), sep = "\t")
qc

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 3.5)
if (is.null(qc) || nrow(qc) == 0 || !"assigned_reads" %in% names(qc)) {
  no_results("Per-sample assigned-read totals not available.")
} else {
  df <- qc |>
    dplyr::mutate(sample    = factor(sample, levels = sample),
                  condition = as.character(condition))
  ggplot(df, aes(x = sample, y = assigned_reads, fill = condition)) +
    geom_col() +
    scale_y_continuous(labels = scales::comma) +
    labs(x = NULL, y = "Assigned reads (sum of counts)") +
    theme_minimal(base_size = 11) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))
}

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 3.5)
if (is.null(qc) || nrow(qc) == 0) {
  no_results("Mapping / duplication metrics not available.")
} else {
  rate_cols <- intersect(c("mapping_rate", "rrna_pct", "duplication"), names(qc))
  rate_cols <- rate_cols[vapply(rate_cols, function(c) any(!is.na(qc[[c]])), logical(1))]
  if (length(rate_cols) == 0) {
    no_results("All rate metrics are NA.")
  } else {
    df <- qc |>
      dplyr::select(sample, condition, dplyr::all_of(rate_cols)) |>
      tidyr::pivot_longer(dplyr::all_of(rate_cols), names_to = "metric", values_to = "value") |>
      dplyr::mutate(sample = factor(sample, levels = unique(sample)))
    ggplot(df, aes(x = sample, y = value, fill = condition)) +
      geom_col() +
      facet_wrap(~ metric, scales = "free_y") +
      labs(x = NULL, y = NULL) +
      theme_minimal(base_size = 11) +
      theme(axis.text.x = element_text(angle = 45, hjust = 1))
  }
}

## 2. Exploratory

VST variable 상위 500개 유전자 사용. 사전 계산된 prcomp 객체를 `results/exploratory/pca.rds`에서 로드.

In [ ]:
pca_obj <- readRDS(rp("results", "exploratory", "pca.rds"))
cor_obj <- readRDS(rp("results", "exploratory", "sample_correlation.rds"))
str(pca_obj, max.level = 1); cat("---\n"); str(cor_obj, max.level = 1)

In [ ]:
options(repr.plot.width = 7.5, repr.plot.height = 5.5)
scores <- pca_obj$scores
ve     <- pca_obj$var_explained
pc1_lbl <- sprintf("PC1 (%.1f%%)", 100 * (ve[["PC1"]] %||% NA))
pc2_lbl <- sprintf("PC2 (%.1f%%)", 100 * (ve[["PC2"]] %||% NA))

ggplot(scores, aes(x = PC1, y = PC2, colour = condition, shape = as.factor(batch), label = sample)) +
  geom_point(size = 3) +
  geom_text(aes(label = sample), nudge_y = 0.04, size = 3, show.legend = FALSE) +
  labs(x = pc1_lbl, y = pc2_lbl, colour = "condition", shape = "batch") +
  theme_minimal(base_size = 12)

In [ ]:
options(repr.plot.width = 7.5, repr.plot.height = 4)
ve <- pca_obj$var_explained
scree <- data.frame(PC = factor(names(ve), levels = names(ve)),
                    pct = as.numeric(ve) * 100)
scree <- head(scree, min(10, nrow(scree)))

ggplot(scree, aes(x = PC, y = pct)) +
  geom_col(fill = "#4c7fb0") +
  labs(x = NULL, y = "% variance explained") +
  theme_minimal(base_size = 12)

In [ ]:
options(repr.plot.width = 7.5, repr.plot.height = 6.5)
M   <- cor_obj$spearman
ord <- if (!is.null(cor_obj$hclust)) cor_obj$hclust$order else seq_len(nrow(M))
M   <- M[ord, ord, drop = FALSE]
df  <- as.data.frame(as.table(M))
names(df) <- c("sample_x", "sample_y", "rho")
df$sample_x <- factor(df$sample_x, levels = rownames(M))
df$sample_y <- factor(df$sample_y, levels = rownames(M))

ggplot(df, aes(x = sample_x, y = sample_y, fill = rho)) +
  geom_tile() +
  geom_text(aes(label = sprintf("%.3f", rho)), size = 3, colour = "grey20") +
  scale_fill_gradient(low = "#ffffff", high = "#4c7fb0",
                      limits = c(min(df$rho, na.rm = TRUE), 1)) +
  labs(x = NULL, y = NULL, fill = expression(rho)) +
  theme_minimal(base_size = 12) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.5)
hc <- cor_obj$hclust
op <- graphics::par(mar = c(6, 1, 1, 1))
plot(hc, main = "", xlab = "", sub = "", ylab = "", yaxt = "n", hang = -1)
graphics::par(op)

## 3. Differential expression — `CONTRAST`

apeglm shrunken LFC, B-H padj. Cutoff는 `cfg$de$primary` / `cfg$de$secondary`에서 가져옴.

In [ ]:
prim <- cfg$de$primary   %||% list(padj = 0.05, abs_lfc = 1)
sec  <- cfg$de$secondary %||% list(padj = 0.10, abs_lfc = 0.5)
prim_padj <- prim$padj; prim_lfc <- prim$abs_lfc
sec_padj  <- sec$padj;  sec_lfc  <- sec$abs_lfc

de_tbl  <- safe_read(rp("results", "de", CONTRAST, "deseq2_results.csv"))
deg_sum <- safe_read(rp("results", "de", CONTRAST, "deg_summary.tsv"), sep = "\t")

dds_path <- rp("results", "exploratory", "dds_vst.rds")
vst_mat  <- if (file.exists(dds_path)) {
  obj <- readRDS(dds_path)
  tryCatch(SummarizedExperiment::assay(obj), error = function(e) as.matrix(obj@assays@data[[1]]))
} else NULL

deg_sum

In [ ]:
options(repr.plot.width = 8.5, repr.plot.height = 6)
v_df <- de_tbl |>
  dplyr::mutate(neglog10p = -log10(pmax(pvalue, 1e-300)),
                sig = dplyr::case_when(
                  is.na(padj)                                        ~ "ns",
                  padj < prim_padj & abs(log2FoldChange) >= prim_lfc ~ "primary",
                  padj < sec_padj  & abs(log2FoldChange) >= sec_lfc  ~ "secondary",
                  TRUE                                               ~ "ns"))
pal <- c(primary = "#c0392b", secondary = "#e67e22", ns = "grey70")

sig_df <- v_df |> dplyr::filter(sig != "ns")
ns_df  <- v_df |> dplyr::filter(sig == "ns")
if (nrow(ns_df) > 3000) { set.seed(42); ns_df <- ns_df[sample.int(nrow(ns_df), 3000), ] }

ggplot() +
  geom_point(data = ns_df,  aes(log2FoldChange, neglog10p),
             colour = "grey70", alpha = 0.45, size = 1.1) +
  geom_point(data = sig_df, aes(log2FoldChange, neglog10p, colour = sig),
             alpha = 0.85, size = 1.6) +
  scale_colour_manual(values = pal) +
  geom_vline(xintercept = c(-prim_lfc, prim_lfc), linetype = "dashed", colour = "grey40") +
  geom_hline(yintercept = -log10(prim_padj),       linetype = "dashed", colour = "grey40") +
  labs(title = CONTRAST, x = "log2FoldChange (apeglm)",
       y = "-log10(pvalue)", colour = "tier") +
  theme_minimal(base_size = 12)

In [ ]:
options(repr.plot.width = 8.5, repr.plot.height = 5.5)
ma_df <- de_tbl |>
  dplyr::mutate(sig = !is.na(padj) & padj < prim_padj & abs(log2FoldChange) >= prim_lfc)

ggplot(ma_df, aes(log10(pmax(baseMean, 1)), log2FoldChange, colour = sig)) +
  geom_point(alpha = 0.5, size = 1.2) +
  scale_colour_manual(values = c(`TRUE` = "#c0392b", `FALSE` = "grey60")) +
  geom_hline(yintercept = 0, colour = "grey40") +
  labs(x = "log10(baseMean)", y = "log2FoldChange (apeglm)",
       colour = sprintf("padj<%.2g & |LFC|>=%.2g", prim_padj, prim_lfc)) +
  theme_minimal(base_size = 12)

In [ ]:
options(repr.plot.width = 8.5, repr.plot.height = 9)
top <- de_tbl |>
  dplyr::filter(!is.na(padj), !is.na(log2FoldChange)) |>
  dplyr::arrange(padj) |>
  head(30)

rows <- intersect(top$gene_id, rownames(vst_mat))
M <- vst_mat[rows, , drop = FALSE]
label_map <- setNames(top$gene_name, top$gene_id)
rn <- ifelse(is.na(label_map[rows]) | !nzchar(label_map[rows]), rows, label_map[rows])
rownames(M) <- make.unique(rn)
Z <- t(scale(t(M)))

row_hc <- hclust(dist(Z),       method = "average")
col_hc <- hclust(dist(t(Z)),    method = "average")
long <- as.data.frame(as.table(Z))
names(long) <- c("gene", "sample", "z")
long$gene   <- factor(long$gene,   levels = rownames(Z)[row_hc$order])
long$sample <- factor(long$sample, levels = colnames(Z)[col_hc$order])

ggplot(long, aes(sample, gene, fill = z)) +
  geom_tile() +
  scale_fill_gradient2(low = "#2c7bb6", mid = "white", high = "#d7191c", midpoint = 0) +
  labs(x = NULL, y = NULL, fill = "z-score") +
  theme_minimal(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        axis.text.y = element_text(size = 9))

In [ ]:
de_tbl |>
  dplyr::arrange(padj) |>
  dplyr::select(dplyr::any_of(c("gene_id", "gene_name", "baseMean",
                                 "log2FoldChange", "lfcSE", "stat",
                                 "pvalue", "padj"))) |>
  head(30)

## 4. GSEA

컬렉션별 Top 10 up / Top 10 down NES 바 플롯, 이어서 지정한 gene set에 대한 running-score enrichment plot.

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 6)
gsea_names <- c(Hallmark     = "gsea_hallmark.csv",
                Reactome     = "gsea_reactome.csv",
                WikiPathways = "gsea_wikipathways.csv",
                PID          = "gsea_pid.csv",
                BioCarta     = "gsea_biocarta.csv",
                CGP          = "gsea_cgp.csv",
                Oncogenic    = "gsea_oncogenic.csv")
gsea_list <- lapply(gsea_names, function(fn)
  safe_read(rp("results", "enrichment", CONTRAST, fn)))
names(gsea_list) <- names(gsea_names)

.trunc <- function(x, n = 50) ifelse(nchar(x) > n, paste0(substr(x, 1, n-1), "..."), x)

for (grp in names(gsea_list)) {
  df <- gsea_list[[grp]]
  if (is.null(df) || nrow(df) == 0) next
  ranked <- df |>
    dplyr::mutate(direction = dplyr::case_when(NES > 0 ~ "up", NES < 0 ~ "down", TRUE ~ NA_character_)) |>
    dplyr::filter(direction %in% c("up", "down")) |>
    dplyr::group_by(direction) |>
    dplyr::arrange(padj, .by_group = TRUE) |>
    dplyr::slice_head(n = 10) |>
    dplyr::ungroup() |>
    dplyr::mutate(label = .trunc(pathway))
  ranked$label <- factor(ranked$label, levels = rev(unique(ranked$label)))
  print(
    ggplot(ranked, aes(NES, label, fill = direction)) +
      geom_col() +
      scale_fill_manual(values = c(up = "#c0392b", down = "#2c7bb6")) +
      geom_vline(xintercept = 0, colour = "grey40") +
      labs(x = "NES", y = NULL,
           title = sprintf("%s — Top 10 up / Top 10 down (by padj)", grp)) +
      theme_minimal(base_size = 11) +
      theme(axis.text.y = element_text(size = 9))
  )
}

### 특정 gene set의 enrichment plot

**setup** 셀을 한 번 실행한 뒤, **plot** 셀은 한 줄짜리 — pathway 이름만 바꾸고 재실행해서 반복. **browse** 셀로 현재 contrast에서 실제로 얻은 pathway 목록을 조회.

랭킹 벡터는 파이프라인이 쓰는 것과 동일한 DE 컬럼(`stat` 또는 `signed_p`)으로 재구성되므로 곡선의 NES/padj는 리포트 값과 일치함.

In [ ]:
## --- GSEA enrichment-plot 헬퍼 (한 번만 실행) ----------------------------
suppressPackageStartupMessages({ library(fgsea); library(msigdbr) })

.ranking <- cfg$enrichment$gsea$ranking %||% "stat"
.species <- cfg$enrichment$species      %||% "Homo sapiens"

.gsea_files <- c(
  H                    = "gsea_hallmark.csv",
  `C2:CP:REACTOME`     = "gsea_reactome.csv",
  `C2:CP:WIKIPATHWAYS` = "gsea_wikipathways.csv",
  `C2:CP:PID`          = "gsea_pid.csv",
  `C2:CP:BIOCARTA`     = "gsea_biocarta.csv",
  `C2:CGP`             = "gsea_cgp.csv",
  C6                   = "gsea_oncogenic.csv"
)

.build_ranks <- function(de_res, method) {
  if (method == "stat") {
    rdf <- de_res |> dplyr::filter(!is.na(gene_name), !is.na(stat)) |>
      dplyr::group_by(gene_name) |> dplyr::summarise(metric = mean(stat), .groups = "drop")
  } else if (method == "signed_p") {
    rdf <- de_res |> dplyr::filter(!is.na(gene_name), !is.na(log2FoldChange), !is.na(pvalue)) |>
      dplyr::mutate(metric = sign(log2FoldChange) * -log10(pmax(pvalue, 1e-300))) |>
      dplyr::filter(is.finite(metric)) |>
      dplyr::group_by(gene_name) |> dplyr::summarise(metric = mean(metric), .groups = "drop")
  } else stop("Unsupported ranking: ", method)
  rdf <- dplyr::arrange(rdf, dplyr::desc(metric))
  setNames(rdf$metric, rdf$gene_name)
}

.get_pathways <- function(coll_str, species) {
  parts    <- unlist(strsplit(coll_str, ":"))
  coll     <- parts[1]
  subcoll  <- if (length(parts) > 1) paste(parts[-1], collapse = ":") else NULL
  m <- if (!is.null(subcoll)) msigdbr(species = species, collection = coll, subcollection = subcoll)
       else                   msigdbr(species = species, collection = coll)
  split(m$gene_symbol, m$gs_name)
}

.pathway_cache <- new.env(parent = emptyenv())

.detect_collection <- function(pathway) {
  # Fast path: MSigDB 관례적 prefix 매칭.
  prefixes <- c(HALLMARK_ = "H", REACTOME_ = "C2:CP:REACTOME",
                WP_ = "C2:CP:WIKIPATHWAYS", PID_ = "C2:CP:PID",
                BIOCARTA_ = "C2:CP:BIOCARTA")
  for (p in names(prefixes)) if (startsWith(pathway, p)) return(unname(prefixes[[p]]))
  # Slow path: 저장된 GSEA CSV 스캔.
  for (coll in names(.gsea_files)) {
    df <- safe_read(rp("results", "enrichment", CONTRAST, .gsea_files[[coll]]))
    if (!is.null(df) && pathway %in% df$pathway) return(coll)
  }
  stop(sprintf("'%s'의 collection을 추론할 수 없음. collection = 인자로 직접 지정하세요.", pathway))
}

.ranks <- .build_ranks(de_tbl, .ranking)

# 공개 헬퍼 -----------------------------------------------------------------

plot_gsea <- function(pathway, collection = NULL) {
  if (is.null(collection)) collection <- .detect_collection(pathway)
  if (is.null(.pathway_cache[[collection]]))
    .pathway_cache[[collection]] <- .get_pathways(collection, .species)
  sets <- .pathway_cache[[collection]]
  if (!pathway %in% names(sets))
    stop(sprintf("'%s'는 %s 컬렉션에 없음.", pathway, collection))
  genes <- sets[[pathway]]
  row   <- safe_read(rp("results", "enrichment", CONTRAST, .gsea_files[[collection]]))
  row   <- if (!is.null(row)) row[row$pathway == pathway, ] else NULL
  nes_padj <- if (!is.null(row) && nrow(row) == 1)
    sprintf(" · NES=%.2f · padj=%.2g", row$NES, row$padj) else ""
  options(repr.plot.width = 8, repr.plot.height = 5)
  fgsea::plotEnrichment(genes, .ranks) +
    ggplot2::labs(title = pathway,
                  subtitle = sprintf("%s · %s · ranking=%s · |set ∩ ranked|=%d%s",
                                     CONTRAST, collection, .ranking,
                                     sum(genes %in% names(.ranks)), nes_padj))
}

list_pathways <- function(query = NULL, n = 20) {
  all <- lapply(names(.gsea_files), function(coll) {
    df <- safe_read(rp("results", "enrichment", CONTRAST, .gsea_files[[coll]]))
    if (is.null(df) || nrow(df) == 0) return(NULL)
    dplyr::mutate(df, collection = coll) |>
      dplyr::select(collection, pathway, NES, padj, size)
  }) |> dplyr::bind_rows()
  if (!is.null(query))
    all <- all[grepl(query, all$pathway, ignore.case = TRUE), ]
  dplyr::arrange(all, padj) |> head(n)
}

show_leading <- function(pathway, collection = NULL) {
  if (is.null(collection)) collection <- .detect_collection(pathway)
  row <- safe_read(rp("results", "enrichment", CONTRAST, .gsea_files[[collection]]))
  row <- row[row$pathway == pathway, ]
  if (nrow(row) == 0) { message("저장된 GSEA 결과에 해당 pathway가 없음."); return(invisible()) }
  le <- strsplit(as.character(row$leadingEdge[1]), ";", fixed = TRUE)[[1]]
  cat(sprintf("%s · %s\nNES=%.3f · padj=%.3g · size=%d · leadingEdge (%d):\n\n",
              pathway, collection, row$NES, row$padj, row$size, length(le)))
  cat(paste(le, collapse = ", "), "\n")
}

invisible(NULL)  # setup 셀은 출력 없이 조용히

In [ ]:
## --- 플롯 (pathway 이름 수정 후 재실행) ----------------------------------
plot_gsea("HALLMARK_INTERFERON_GAMMA_RESPONSE")

# Collection은 이름 prefix로 자동 감지. 모호할 때는 명시:
# plot_gsea("MY_CUSTOM_SET", collection = "C2:CGP")

In [ ]:
## --- 현재 contrast에서 실제 나온 pathway 조회 / 검색 --------------------
# 모든 컬렉션 통합 Top 20 (padj 순):
list_pathways()

# 키워드 필터 (대소문자 무시):
# list_pathways("interferon")
# list_pathways("apop", n = 50)

# 특정 pathway의 leading-edge 유전자:
# show_leading("HALLMARK_INTERFERON_GAMMA_RESPONSE")

## 5. ORA

`ora_combined.csv` 기준 데이터베이스별 Top 10 up / Top 10 down (p.adjust 순).

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 7)
ora <- safe_read(rp("results", "enrichment", CONTRAST, "ora_combined.csv"))
if (is.null(ora) || nrow(ora) == 0) {
  no_results("ORA returned no results.")
} else {
  for (db in unique(ora$database)) {
    show_df <- ora |>
      dplyr::filter(database == db, direction %in% c("up", "down")) |>
      dplyr::group_by(direction) |>
      dplyr::arrange(p.adjust, .by_group = TRUE) |>
      dplyr::slice_head(n = 10) |>
      dplyr::ungroup() |>
      dplyr::mutate(
        neglog10p   = -log10(pmax(p.adjust, 1e-300)),
        Description = ifelse(is.na(Description) | !nzchar(Description), ID, Description),
        Description = .trunc(Description, 50))
    if (nrow(show_df) == 0) next
    print(
      ggplot(show_df, aes(neglog10p, reorder(Description, neglog10p), fill = direction)) +
        geom_col(position = position_dodge(width = 0.8)) +
        scale_fill_manual(values = c(up = "#c0392b", down = "#2c7bb6")) +
        labs(x = "-log10(p.adjust)", y = NULL,
             title = sprintf("%s — Top 10 up / Top 10 down", db)) +
        theme_minimal(base_size = 11)
    )
  }
}

## 6. TF activity (CollecTRI / ULM)

`results/tf_activity/<contrast>/tf_top.tsv`에서 |score| 상위 TF.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 8)
tf <- safe_read(rp("results", "tf_activity", CONTRAST, "tf_top.tsv"), sep = "\t")
if (is.null(tf) || nrow(tf) == 0) {
  no_results("TF activity results not available.")
} else {
  top <- tf |> dplyr::slice_max(abs(score), n = 30) |>
    dplyr::mutate(tf = forcats::fct_reorder(source, score))
  ggplot(top, aes(score, tf, fill = score > 0)) +
    geom_col() +
    scale_fill_manual(values = c(`TRUE` = "#c0392b", `FALSE` = "#2c7bb6"), guide = "none") +
    labs(title = paste("Top TF activity —", CONTRAST), y = NULL) +
    theme_minimal(base_size = 11)
}

## 7. Drug repositioning (L2S2)

`results/drug_repositioning/<contrast>/l2s2_hits.tsv`에서 |score| 상위 perturbagen.

In [ ]:
options(repr.plot.width = 8.5, repr.plot.height = 8)
hits <- safe_read(rp("results", "drug_repositioning", CONTRAST, "l2s2_hits.tsv"), sep = "\t")
if (is.null(hits) || nrow(hits) == 0) {
  no_results("L2S2 query returned no hits.")
} else {
  top <- hits |>
    dplyr::arrange(dplyr::desc(abs(score %||% 0)), fdr) |>
    head(30) |>
    dplyr::mutate(label = ifelse(is.na(perturbagen) | !nzchar(perturbagen),
                                 paste0("row_", dplyr::row_number()), perturbagen),
                  label = factor(label, levels = rev(label)))
  ggplot(top, aes(score, label, fill = ifelse(score > 1, "induce", "reverse"))) +
    geom_col() +
    scale_fill_manual(values = c(induce = "#e67e22", reverse = "#2c7bb6"), name = "direction") +
    geom_vline(xintercept = 1, linetype = "dashed", colour = "grey40") +
    labs(x = "odds ratio (score)", y = NULL, title = "Top 30 perturbagens") +
    theme_minimal(base_size = 11)
}